#多维数组的输入

In [9]:
fruits = ['apple', 'banana', 'cherry']

# 传统写法（需手动计数）
count = 0
for fruit in fruits:
    print(count, fruit)
    count += 1

# 使用 enumerate（推荐）
for index, fruit in enumerate(fruits):
    print(index, fruit)  # 输出：0 apple → 1 banana → 2 cherry

# 设置索引起始值
for index, fruit in enumerate(fruits, start=1):
    print(index, fruit)  # 输出：1 apple → 2 banana → 3 cherry

0 apple
1 banana
2 cherry
0 apple
1 banana
2 cherry
1 apple
2 banana
3 cherry


In [28]:
# 特征字段
import torch


# 准备数据
import numpy  as np

# x= np.loadtxt("diabetes_data_raw.csv.gz",delimiter=' ',dtype=np.float32)
xy = np.loadtxt('diabetes.csv.gz',delimiter= ',',dtype=np.float32)
x_data = torch.from_numpy(xy[:,:-1])
y_data = torch.from_numpy(xy[:,[-1]])



# 构建模型
class Model(torch.nn.Module):
    def __init__(self):
        super(Model,self).__init__()
        self.linear1 = torch.nn.Linear(8,6)
        self.linear2 = torch.nn.Linear(6,4)
        self.linear3 = torch.nn.Linear(4,1)
        self.sigmoid = torch.nn.Sigmoid()
    def forward(self,x):
        x = self.sigmoid(self.linear1(x))
        x= self.sigmoid(self.linear2(x))
        x = self.sigmoid(self.linear3(x))
        
        return x
model = Model()

# 构建损失函数与优化器

criterion = torch.nn.BCELoss(size_average=False)

optimizer = torch.optim.SGD(model.parameters(),lr=0.011)


# 训练循环

for epoch in range(10000):
    # 前向模型
    y_pred = model(x_data)
    loss = criterion(y_pred,y_data)
    print(epoch,loss.item())
    optimizer.zero_grad()
    loss.backward()
    
    optimizer.step()


0 506.30596923828125
1 601.5503540039062
2 845.6712646484375
3 987.4293823242188
4 490.2244873046875
5 490.13543701171875
6 490.03948974609375
7 489.9556579589844
8 489.86688232421875
9 489.78594970703125
10 489.70111083984375
11 489.62078857421875
12 489.5369567871094
13 489.455078125
14 489.3694763183594
15 489.2837219238281
16 489.193603515625
17 489.10137939453125
18 489.0037841796875
19 488.90216064453125
20 488.7938537597656
21 488.6793518066406
22 488.5563049316406
23 488.42474365234375
24 488.2821044921875
25 488.1280517578125
26 487.9596252441406
27 487.7760009765625
28 487.57373046875
29 487.3512268066406
30 487.10418701171875
31 486.8302917480469
32 486.5239562988281
33 486.1815185546875
34 485.795654296875
35 485.3611755371094
36 484.8680419921875
37 484.3089294433594
38 483.66998291015625
39 482.94146728515625
40 482.1041564941406
41 481.1454772949219
42 480.03924560546875
43 478.7713317871094
44 477.3064880371094
45 475.634033203125
46 473.70806884765625
47 471.5374755859

In [29]:
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# 1. 数据加载与预处理
xy = np.loadtxt('diabetes.csv.gz', delimiter=',', dtype=np.float32)
x_data = xy[:, :-1]
y_data = xy[:, [-1]]

# 标准化特征数据 (关键步骤!)
scaler = StandardScaler()
x_data = scaler.fit_transform(x_data)

# 转换为PyTorch张量
x_data = torch.from_numpy(x_data).float()
y_data = torch.from_numpy(y_data).float()

# 2. 创建数据集和数据加载器 (使用小批量训练)
dataset = TensorDataset(x_data, y_data)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

# 3. 改进模型结构
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        self.layer1 = nn.Linear(8, 16)
        self.layer2 = nn.Linear(16, 8)
        self.layer3 = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(0.2)  # 添加Dropout防止过拟合
        
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.dropout(x)
        x = self.layer3(x)  # 注意：最后一层不使用激活函数
        return x

model = DiabetesModel()

# 4. 使用更合适的损失函数和优化器
criterion = nn.BCEWithLogitsLoss()  # 比BCELoss更稳定，包含sigmoid
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # 自适应学习率
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, verbose=True
)

# 5. 训练循环 (使用小批量训练)
for epoch in range(500):  # 减少epoch次数但更高效
    epoch_loss = 0.0
    model.train()
    
    for inputs, labels in train_loader:
        # 前向传播
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * inputs.size(0)
    
    # 计算平均epoch损失
    epoch_loss /= len(dataset)
    
    # 学习率调整
    scheduler.step(epoch_loss)
    
    # 每10个epoch打印一次
    if epoch % 10 == 0:
        print(f'Epoch [{epoch+1}/500], Loss: {epoch_loss:.4f}')
    
    # 提前停止条件
    if epoch_loss < 0.45:  # 设置合理的损失阈值
        print(f"训练提前停止，达到低损失值: {epoch_loss:.4f}")
        break

# 6. 模型评估
model.eval()
with torch.no_grad():
    # 计算训练集准确率
    outputs = model(x_data)
    predicted = torch.sigmoid(outputs) > 0.5
    correct = (predicted.float() == y_data).sum().item()
    accuracy = correct / len(y_data) * 100
    
    # 计算最终损失
    final_loss = criterion(outputs, y_data).item()
    
    print(f"最终损失值: {final_loss:.4f}")
    print(f"训练集准确率: {accuracy:.2f}%")

# 7. 保存模型
torch.save(model.state_dict(), 'diabetes_model.pth')
print("模型已保存!")

d:\Users\zyl\anaconda3\envs\pytorchgpu\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/500], Loss: 0.6968
Epoch [11/500], Loss: 0.5365
Epoch [21/500], Loss: 0.4888
Epoch [31/500], Loss: 0.4761
Epoch [41/500], Loss: 0.4843
Epoch [51/500], Loss: 0.4582
Epoch [61/500], Loss: 0.4715
Epoch [71/500], Loss: 0.4649
Epoch [81/500], Loss: 0.4625
训练提前停止，达到低损失值: 0.4437
最终损失值: 0.4411
训练集准确率: 77.60%
模型已保存!


#MNIST dataset



In [ ]:
# 1）__init __方法的第一参数永远是self，表示创建的类实例本身，因此，在__init __方法内部，就可以把各种属性绑定到self，因为self就指向创建的实例本身。
# （2）有了__init __方法，在创建实例的时候，就不能传入空的参数了，必须传入与__init __方法匹配的参数，但self不需要传，Python解释器会自己把实例变量传进去：
# 这样一来，我们从外部看Student类，就只需要知道，创建实例需要给出name和score。而如何打印，都是在Student类的内部定义的，这些数据和逻辑被封装起来了，调用很容易，但却不知道内部实现的细节。
# 如果要让内部属性不被外部访问，可以把属性的名称前加上两个下划线，在Python中，实例的变量名如果以__开头，就变成了一个私有变量（private），只有内部可以访问，外部不能访问，所以，我们把Student类改一改：
# 另外，这里self 就是指实例本身，self.name就是Student类的属性变量，是Student类所有。而name是外部传来的参数，不是Student类所自带的。故，self.name = name的意思就是把外部传来的参数name的值赋值给Student类自己的属性变量self.name。
# class Test:
#     def ppr(self):
#         print(self)
#         print(self.__class__)

# t = Test()
# t.ppr()
# 执行结果：
# <__main__.Test object at 0x000000000284E080>
# <class '__main__.Test'>

# 从上面的例子中可以很明显的看出，self代表的是类的实例。而 self.__ class __ 则指向类。
# 注意：把self换成this，结果也一样，但Python中最好用约定俗成的self。

# （2）self可以不写吗？
# 在Python解释器的内部，当我们调用t.ppr()时，实际上Python解释成Test.ppr(t)，也就是把self替换成了类的实例。
# 运行时提醒错误如下：ppr在定义时没有参数，但是我们运行时强行传了一个参数。

# 由于上面解释过了t.ppr()等同于Test.ppr(t)，所以程序提醒我们多传了一个参数t。

# 这里实际上已经部分说明了 self 在定义时不可以省略。

# 当然，如果我们的定义和调用时均不传类实例是可以的，这就是类方法。



# 魔法方法就是：

# Python 在类里规定好名字、自动调用的方法
# 你只要按照语法实现好，它会在特定时机被自动触发
# 通常以 __xxx__ 的形式命名，比如 __init__, __len__, __add__, __getitem__

In [ ]:
class Cat:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return f"  这是一只叫 {self.name} 的猫"

c = Cat("咪咪")
print(c)  # 自动调用 __str__()




class Price:
    def __init__(self, yuan):  # 构造函数，初始化对象
        self.yuan = yuan        # 将传入的元数值存储为实例属性

    def __add__(self, other):   # 重载加法运算符 (+)
        return Price(self.yuan + other.yuan)  # 创建新对象，值为两对象yuan之和

    def __eq__(self, other):    # 重载相等运算符 (==)
        return self.yuan == other.yuan  # 比较两对象的yuan值是否相等

    def __str__(self):          # 重载字符串转换方法 (print/str时自动调用)
        return f"{self.yuan}元"  # 返回格式化的字符串（如 "50元"）

# 实例化两个对象
p1 = Price(30)  # 创建 yuan=30 的Price对象
p2 = Price(20)  # 创建 yuan=20 的Price对象

print(p1 + p2)  # 等价于 print(Price(30+20)) → 触发__str__ → 输出"50元"
print(p1 == p2) # 等价于 print(30 == 20) → 返回False


# 函数重载允许我们定义多个同名函数，这些函数根据传入参数的数量或类型不同执行不同的逻辑。这在静态类型语言中较为常见，有助于提高代码的可读性和灵活性。



# 二、为什么要用装饰器
# 使用装饰器之前，我们要知道，其实python里是万物皆对象，也就是万物都可传参。

# 函数也可以作为函数的参数进行传递的。

  这是一只叫 咪咪 的猫


In [ ]:
def baiyu():
    print("我是攻城狮白玉")
 
 
def blog(name):
    print('进入blog函数')
    name()
    print('我的博客是 https://blog.csdn.net/zhh763984017')
 
 
if __name__ == '__main__':
    func = baiyu  # 这里是把baiyu这个函数名赋值给变量func
    func()  # 执行func函数
    print('------------')
    blog(baiyu)  # 把baiyu这个函数作为参数传递给blog函数
    
    
    
    import time
 
 
def baiyu():
    t1 = time.time()
    print("我是攻城狮白玉")
    time.sleep(2)
    print("执行时间为：", time.time() - t1)
 
 
def blog(name):
    t1 = time.time()
    print('进入blog函数')
    name()
    print('我的博客是 https://blog.csdn.net/zhh763984017')
    print("执行时间为：", time.time() - t1)
 
 
if __name__ == '__main__':
    func = baiyu  # 这里是把baiyu这个函数名赋值给变量func
    func()  # 执行func函数
    print('------------')
    blog(baiyu)  # 把baiyu这个函数作为参数传递给blog函数
    
    
    
    
    import time
 
 
def baiyu():
    print("我是攻城狮白玉")
    time.sleep(2)
 
 
def count_time(func):
    def wrapper():
        t1 = time.time()
        func()
        print("执行时间为：", time.time() - t1)
 
    return wrapper
 
 
if __name__ == '__main__':
    baiyu = count_time(baiyu)  # 因为装饰器 count_time(baiyu) 返回的是函数对象 wrapper，这条语句相当于  baiyu = wrapper
    baiyu()  # 执行baiyu()就相当于执行wrapper()
    
    
    
    import time
 
 
def count_time(func):
    def wrapper():
        t1 = time.time()
        func()
        print("执行时间为：", time.time() - t1)
 
    return wrapper
 
 
@count_time
def baiyu():
    print("我是攻城狮白玉")
    time.sleep(2)
 
 
if __name__ == '__main__':
    # baiyu = count_time(baiyu)  # 因为装饰器 count_time(baiyu) 返回的时函数对象 wrapper，这条语句相当于  baiyu = wrapper
    # baiyu()  # 执行baiyu()就相当于执行wrapper()
 
    baiyu()  # 用语法糖之后，就可以直接调用该函数了

我是攻城狮白玉
------------
进入blog函数
我是攻城狮白玉
我的博客是 https://blog.csdn.net/zhh763984017
我是攻城狮白玉
执行时间为： 2.008059024810791
------------
进入blog函数
我是攻城狮白玉
执行时间为： 2.0069220066070557
我的博客是 https://blog.csdn.net/zhh763984017
执行时间为： 2.0069220066070557


In [18]:
import torch
import numpy as np
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
from torch.utils.data import Dataset


class DiabetesDataset(Dataset):
    def __init__(self,filepath):
        super().__init__()
        xy = np.loadtxt(filepath,delimiter=',',dtype=np.float32)
        self.len =xy.shape[0]
        self.x_data = torch.from_numpy(xy[:,:-1])
        self.y_data = torch.from_numpy(xy[:,[-1]])
    def __getitem__(self, index):   #the expression,dataset[index],will call this magic function
      return self.x_data[index],self.y_data[index]
    def __len__(self):   #this magic function returns length of dataset
        return self.len
    #dataloader initializer loader with batch-size,shuffle,process number
    #  num_workers=2#多线程
dataset = DiabetesDataset('diabetes.csv.gz')
train_loader = DataLoader(dataset=dataset,  batch_size=32,shuffle=True, num_workers=2)
                          
# 构建模型
class Model(torch.nn.Module):
    def __init__(self):
        super(Model,self).__init__()
        self.linear1 = torch.nn.Linear(8,6)
        self.linear2 = torch.nn.Linear(6,4)
        self.linear3 = torch.nn.Linear(4,1)
        self.sigmoid = torch.nn.Sigmoid()
    def forward(self,x):
        x = self.sigmoid(self.linear1(x))
        x= self.sigmoid(self.linear2(x))
        x = self.sigmoid(self.linear3(x))
        
        return x
                         
model = Model()

# 构建损失函数与优化器

criterion = torch.nn.BCELoss(size_average=False)

optimizer = torch.optim.SGD(model.parameters(),lr=0.1)
 
                          
# the implementation of multiprocessing is different on windows,which uses spawn instead of fork
# so we have to wrap the code with an if-clause to protect the code from executon multiple tmes
if __name__ == '_main_':
    for epoch in range(100):
        for i,data in enumerate(train_loader,0):
            # 1.prepare data 
            inputs,labels =data
            #forward 
            y_pred = model(inputs)
            loss = criterion(y_pred,labels)
            print(epoch,i,loss.item())
            optimizer.zero_grad()
            loss.backward()
            # update
            optimizer.step()
# train_dataset = datasets.MNIST(root='../dataset/mnist')



#training cycle
# definiton :one forward pass and one backward pass of the the training examples
# batch-size :the number of training examples in one forward backward pass
# ilteration (迭代): number of passes,each pass using (batch size) number of examples
# for epoch in range (training_epochs):
#     #LOOP循环 over all batches
#     for i in range(total_batch):
        